In [8]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Policy Network (2 hidden layers, ReLU)
class PolicyNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_actions):
        super(PolicyNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, num_actions)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)


# 2. Car Environment
class CarEnv:
    """
    Locked state/action spec:
    - State: [x, y, vx, vy, s1, s2, ..., sk]
    - Actions: 0=turn_left, 1=turn_right, 2=speed_up, 3=no_action
    """

    def __init__(self, num_sensors=5, max_steps=500):
        assert num_sensors in {1, 3, 5, 7, 9}, "num_sensors must be one of {1,3,5,7,9}"
        self.num_sensors = num_sensors
        self.max_steps = max_steps

        # Competition parameters
        self.min_speed = 0.001
        self.max_speed = 0.1
        self.turn_delta = 0.01
        self.turn_speed_factor = 0.9875   # -1.25%
        self.speedup_factor = 1.025       # +2.5%
        self.crash_speed_factor = 0.125   # reduce to 12.5%

        # Sensor fan in front of the car
        self.sensor_fov = math.pi / 2.0
        self.sensor_max_range = math.sqrt(2.0)

        self.reset()

    def reset(self):
        self.x = 0.5
        self.y = 0.5
        self.angle = 0.0
        self.speed = 0.01
        self.step_count = 0
        self.crash_count = 0
        self.total_distance = 0.0
        return self._get_state()

    def _ray_distance_to_walls(self, ray_angle):
        dx = math.cos(ray_angle)
        dy = math.sin(ray_angle)
        eps = 1e-12
        candidates = []

        if abs(dx) > eps:
            t = (0.0 - self.x) / dx
            if t >= 0:
                y_hit = self.y + t * dy
                if 0.0 <= y_hit <= 1.0:
                    candidates.append(t)
            t = (1.0 - self.x) / dx
            if t >= 0:
                y_hit = self.y + t * dy
                if 0.0 <= y_hit <= 1.0:
                    candidates.append(t)

        if abs(dy) > eps:
            t = (0.0 - self.y) / dy
            if t >= 0:
                x_hit = self.x + t * dx
                if 0.0 <= x_hit <= 1.0:
                    candidates.append(t)
            t = (1.0 - self.y) / dy
            if t >= 0:
                x_hit = self.x + t * dx
                if 0.0 <= x_hit <= 1.0:
                    candidates.append(t)

        if not candidates:
            return self.sensor_max_range
        return min(candidates)

    def _get_sensor_readings(self):
        if self.num_sensors == 1:
            angles = [self.angle]
        else:
            left = self.angle - self.sensor_fov / 2.0
            step = self.sensor_fov / (self.num_sensors - 1)
            angles = [left + i * step for i in range(self.num_sensors)]

        readings = []
        for ray_angle in angles:
            d = self._ray_distance_to_walls(ray_angle)
            readings.append(min(d / self.sensor_max_range, 1.0))
        return readings

    def _get_state(self):
        vx = self.speed * math.cos(self.angle)
        vy = self.speed * math.sin(self.angle)
        return np.array([self.x, self.y, vx, vy] + self._get_sensor_readings(), dtype=np.float32)

    def step(self, action):
        self.step_count += 1

        # Action effects
        if action == 0:
            self.angle -= self.turn_delta
            self.speed *= self.turn_speed_factor
        elif action == 1:
            self.angle += self.turn_delta
            self.speed *= self.turn_speed_factor
        elif action == 2:
            self.speed *= self.speedup_factor
        elif action == 3:
            pass
        else:
            raise ValueError("action must be in {0,1,2,3}")

        self.speed = max(self.min_speed, min(self.speed, self.max_speed))

        old_x, old_y = self.x, self.y
        new_x = self.x + self.speed * math.cos(self.angle)
        new_y = self.y + self.speed * math.sin(self.angle)

        crashed = (new_x < 0.0 or new_x > 1.0 or new_y < 0.0 or new_y > 1.0)
        if crashed:
            self.crash_count += 1
            self.speed *= self.crash_speed_factor
            self.speed = max(self.min_speed, min(self.speed, self.max_speed))

        self.x = min(max(new_x, 0.0), 1.0)
        self.y = min(max(new_y, 0.0), 1.0)

        frame_distance = math.sqrt((self.x - old_x) ** 2 + (self.y - old_y) ** 2)
        self.total_distance += frame_distance

        done = self.step_count >= self.max_steps
        info = {
            "frame_distance": frame_distance,
            "crashed": crashed,
            "crash_count": self.crash_count,
            "total_distance": self.total_distance,
            "speed": self.speed,
        }
        return self._get_state(), frame_distance, done, info


def generate_trajectory(network, env):
    trajectory = []
    state = env.reset()
    done = False

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        logits = network(state_tensor)

        probs = torch.softmax(logits, dim=0)
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample().item()

        next_state, reward, done, info = env.step(action)

        trajectory.append({
            "state": state.copy(),
            "action": action,
            "reward": reward,
            "info": info,
        })
        state = next_state

    return trajectory


def train_reinforce(network, optimizer, trajectories):
    optimizer.zero_grad()
    total_loss = 0

    for trajectory in trajectories:
        total_reward = sum(step["reward"] for step in trajectory)

        for step in trajectory:
            state_tensor = torch.tensor(step["state"], dtype=torch.float32)
            action_taken = step["action"]

            action_scores = network(state_tensor)
            action_probs = torch.softmax(action_scores, dim=0)

            log_prob = torch.log(action_probs[action_taken] + 1e-8)
            total_loss -= log_prob * total_reward

    total_loss.backward()
    optimizer.step()


def evaluate_policy(network, env, episodes=10):
    distance_scores = []
    crash_scores = []

    for _ in range(episodes):
        state = env.reset()
        done = False

        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32)
            with torch.no_grad():
                scores = network(state_tensor)
                action = torch.argmax(scores).item()
            state, _, done, _ = env.step(action)

        distance_scores.append(env.total_distance)
        crash_scores.append(env.crash_count)

    print(f"Avg distance over {episodes} episodes: {np.mean(distance_scores):.4f}")
    print(f"Avg crashes over {episodes} episodes: {np.mean(crash_scores):.2f}")

In [9]:
# 1. Choose sensor count k (must be odd in {1,3,5,7,9})
k = 7
input_size = 4 + k  # [x, y, vx, vy] + k sensors

# 2. Create environment and network
env = CarEnv(num_sensors=k, max_steps=500)
net = PolicyNetwork(input_size=input_size, hidden_size=64, num_actions=4)

# 3. Create optimizer
optimizer = optim.Adam(net.parameters(), lr=0.01)

print("Environment and policy initialized")
print("State size:", input_size)
print("Action mapping: 0=left, 1=right, 2=speed_up, 3=no_action")

Environment and policy initialized
State size: 11
Action mapping: 0=left, 1=right, 2=speed_up, 3=no_action


In [10]:
# --- Real environment sanity check ---
state0 = env.reset()
print("Initial state length:", len(state0), "expected:", 4 + k)
print("Initial state:", state0)

state_tensor = torch.tensor(state0, dtype=torch.float32)
with torch.no_grad():
    scores = net(state_tensor)
    probs = torch.softmax(scores, dim=0)

print("Network output shape:", tuple(scores.shape))
print("Action probabilities:", probs.numpy())
print("Probability sum:", float(probs.sum()))

action_det = torch.argmax(scores).item()
state1, reward1, done1, info1 = env.step(action_det)
print("After one step -> action:", action_det)
print("Reward (frame distance):", reward1)
print("Done:", done1)
print("Crash this step:", info1["crashed"], "Total crashes:", info1["crash_count"])
print("Total distance so far:", info1["total_distance"])

Initial state length: 11 expected: 11
Initial state: [0.5        0.5        0.01       0.         0.5        0.4082483
 0.36602542 0.35355338 0.36602542 0.4082483  0.5       ]
Network output shape: (4,)
Action probabilities: [0.27471727 0.25453562 0.26375082 0.20699634]
Probability sum: 1.0
After one step -> action: 0
Reward (frame distance): 0.009874999999999998
Done: False
Crash this step: False Total crashes: 0
Total distance so far: 0.009874999999999998
